# Load Data

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split

#eval dataset
df = pd.read_csv("C://Users//grete//Documents//20260810_model_benchmarking//rater123_finallabel2.csv")

#grouped = df.groupby(['labels'])
grouped = df.groupby(by='final_rating')
#
grouped_data = pd.concat([grouped.get_group(x) for x in grouped.groups])

#train_data, test_validation_data = train_test_split(df, test_size=0.26, random_state=10, stratify=df['labels'])
train_data, test_validation_data = train_test_split(df, test_size=0.26, random_state=10, stratify=df['final_rating'])
#test_data, validation_data = train_test_split(test_validation_data, test_size=0.25, random_state=10, stratify=test_validation_data['labels'])
test_data, validation_data = train_test_split(test_validation_data, test_size=0.25, random_state=10, stratify=test_validation_data['final_rating'])

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)
validation_df = pd.DataFrame(validation_data)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(validation_df)}")

# Check label distribution in each split
for split_name, split_df in zip(
    ["train", "test", "validation"],
    [train_df, test_df, validation_df]
):
#    print(f"\n{split_name.upper()} labels:")
    print(f"\n{split_name.upper()} final_rating:")
#    print(split_df['labels'].value_counts())
    print(split_df['final_rating'].value_counts())

Train size: 1172, Test size: 309, Validation size: 103

TRAIN final_rating:
final_rating
0    589
1    583
Name: count, dtype: int64

TEST final_rating:
final_rating
0    155
1    154
Name: count, dtype: int64

VALIDATION final_rating:
final_rating
0    52
1    51
Name: count, dtype: int64


In [2]:
#Set to test df for eval 

df = test_df
#texts = df["sentence-chatgpt"].tolist()
texts = df["sentence_gpt4"].tolist()
#true_labels = df["description"].tolist()
true_labels = df["final_rating"].tolist()

print(f"Loaded {len(texts)} samples")
print(df.columns.tolist())

Loaded 309 samples
['normalizedGenre', 'filename', 'play_title', 'speaker_id', 'speaker_name', 'speaker_spblock', 'gender', 'sentence_number', 'sample_index', 'sentence_original', 'sentence_gpt4', 'rater1', 'rater2', 'rater3', 'final_rating', 'rating_match']


In [3]:
# ============================================================
# Summary Metrics
# ============================================================
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
# added 20260810
from sklearn.utils import resample
#
import numpy as np

def bootstrap_metrics(true, pred, n=1000):
    f1s, accs, precs, recs = [], [], [], []
    for _ in range(n):
        idx = resample(range(len(true)), random_state=None)
        t = [true[i] for i in idx]
        p = [pred[i] for i in idx]
        f1s.append(f1_score(t, p, average="macro",  zero_division=0))
        accs.append(accuracy_score(t, p))
        precs.append(precision_score(t, p, average="macro", zero_division=0))
        recs.append(recall_score(t, p, average="macro",     zero_division=0))

    def ci(vals):
        lo, hi = np.percentile(vals, [2.5, 97.5])
        return np.mean(vals), lo, hi

    return {
        "f1_macro":  ci(f1s),
        "accuracy":  ci(accs),
        "precision": ci(precs),
        "recall":    ci(recs),
    }


def evaluate_model(name, true, pred):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(classification_report(
        true, pred,
        target_names=["non-description (0)", "description (1)"],
        digits=3
    ))
    print("Confusion matrix (rows=true, cols=pred):")
    cm = confusion_matrix(true, pred)
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

    ci = bootstrap_metrics(true, pred)
    print("\n95% Bootstrap Confidence Intervals (1000 resamples):")
    for metric, (mean, lo, hi) in ci.items():
        print(f"  {metric:<12}: {mean:.3f}  [{lo:.3f} – {hi:.3f}]")

    return {
        "model":      name,
        "accuracy":   accuracy_score(true, pred),
        "f1_macro":   f1_score(true, pred, average="macro",  zero_division=0),
        "f1_class0":  f1_score(true, pred, pos_label=0,      zero_division=0), 
        "f1_class1":  f1_score(true, pred, pos_label=1,      zero_division=0),
        "precision":  precision_score(true, pred, average="macro", zero_division=0),
        "recall":     recall_score(true, pred, average="macro",    zero_division=0),
        "ci_f1_lo":   ci["f1_macro"][1],
        "ci_f1_hi":   ci["f1_macro"][2],
    }

all_results = []

# Hartmann DistilRoberta, SamLowe/Goemotions

In [4]:
import torch 
from transformers import pipeline
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# HARTMANN DistilRoBERTa EMOTION
# ============================================================
# Labels: anger, disgust, fear, joy, neutral, sadness, surprise
# Mapping: neutral → 0, all other emotions → 1

print("\n[3/5] Loading Hartmann DistilRoBERTa Emotion...")

hartmann_pipe = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=DEVICE
)

def predict_emotion_model(texts, pipe, neutral_label="neutral", batch_size=16):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        results = pipe(batch)
        for r in results:
            preds.append(0 if r["label"] == neutral_label else 1)
    return preds

hartmann_preds = predict_emotion_model(texts, hartmann_pipe)
all_results.append(evaluate_model("Hartmann DistilRoBERTa Emotion", true_labels, hartmann_preds))

del hartmann_pipe
torch.cuda.empty_cache() if torch.cuda.is_available() else None

C:\Users\grete\anaconda3\envs\env_nltk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[3/5] Loading Hartmann DistilRoBERTa Emotion...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18368.45it/s]



  Hartmann DistilRoBERTa Emotion
                     precision    recall  f1-score   support

non-description (0)      0.790     0.606     0.686       155
    description (1)      0.679     0.838     0.750       154

           accuracy                          0.722       309
          macro avg      0.734     0.722     0.718       309
       weighted avg      0.735     0.722     0.718       309

Confusion matrix (rows=true, cols=pred):
  TN=94  FP=61
  FN=25  TP=129

95% Bootstrap Confidence Intervals (1000 resamples):
  f1_macro    : 0.717  [0.664 – 0.769]
  accuracy    : 0.721  [0.670 – 0.770]
  precision   : 0.734  [0.682 – 0.784]
  recall      : 0.722  [0.673 – 0.771]


In [5]:
# ============================================================
# SamLowe RoBERTa GoEmotions (28 labels)
# ============================================================
# Mapping: neutral → 0, all other emotions → 1

print("\n[4/5] Loading SamLowe RoBERTa GoEmotions...")

samlowe_pipe = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    device=DEVICE
)

samlowe_preds = predict_emotion_model(texts, samlowe_pipe, neutral_label="neutral")
all_results.append(evaluate_model("SamLowe RoBERTa GoEmotions", true_labels, samlowe_preds))

del samlowe_pipe
torch.cuda.empty_cache() if torch.cuda.is_available() else None


[4/5] Loading SamLowe RoBERTa GoEmotions...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11351.22it/s]



  SamLowe RoBERTa GoEmotions
                     precision    recall  f1-score   support

non-description (0)      0.639     0.697     0.667       155
    description (1)      0.664     0.604     0.633       154

           accuracy                          0.650       309
          macro avg      0.652     0.650     0.650       309
       weighted avg      0.652     0.650     0.650       309

Confusion matrix (rows=true, cols=pred):
  TN=108  FP=47
  FN=61  TP=93

95% Bootstrap Confidence Intervals (1000 resamples):
  f1_macro    : 0.648  [0.593 – 0.698]
  accuracy    : 0.649  [0.595 – 0.699]
  precision   : 0.650  [0.597 – 0.701]
  recall      : 0.649  [0.595 – 0.699]


# Gemma 4 via Ollama

In [6]:
# ============================================================
# Gemma 4 using Ollama
# ============================================================

import ollama
import pandas as pd

MODEL_NAME = "gemma4:e4b" #slow on GPU

PROMPT_2 = """
I am attaching sentences from early modern British plays. You are a specialist in early modern British plays (c. 1500–1700). 
Your task is to Label each sentence according to whether it contains a description of an emotion according to the following 3 rules:
1. Label 1 for sentences explicitly describes an emotional state, e.g. "she felt grief", "he was seized with joy". 
2. Label 1 for sentences describing or metaphorically expressing an emotional state without naming an emotion, 
e.g. "piercing his heart", "passion overruling him".
3. Label 0 for sentences that do not describe an emotion, even if they depict an emotionally charged event or situation.
The examples in the rules are non-exhaustive. 
Return only the label 1 or 0. Do not output anything else, no explanation."""

# ── Prompt builder ───────────────────────────────────────────────
def build_messages(text, few_shot=False):
    messages = [{"role": "system", "content": PROMPT_2}]
    #if few_shot:
        #for example in FEW_SHOT_EXAMPLES:
            #messages.append({"role": "user",      "content": example["user"]})
            #messages.append({"role": "assistant", "content": example["assistant"]})
    messages.append({"role": "user", "content": text.strip()})

    return messages
    
# ── Predict ──────────────────────────────────────────────────────
df_results = df.copy()
preds, raws = [], []

def predict(texts, few_shot=False):
    for i, text in enumerate(texts): 
        messages = build_messages(text, few_shot=few_shot)
        #"think": False,                    # explicit API-level disable
        #"stream": False,
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
                #"num_predict": 2,       # equivalent to max_new_tokens=2
                "temperature": 0.0,     # deterministic output
            }
        )
        # full raw response content
        raws.append(response)
        # parse 0/1 from response
        answer = response["message"].content

        preds.append(
            1 if answer.startswith("1") else
            0 if answer.startswith("0") else
            (1 if "1" in answer[:10] else 0)
        )
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{len(texts)}")
    return preds, raws

# Main Loop
preds, raws = predict(texts, few_shot=False)

df_results.loc[df_results.index[:len(preds)], "preds"] = preds
df_results.loc[df_results.index[:len(raws)], "raws"] = raws

df_results.to_csv("gemma4-e4b_P2fewshot.csv", encoding="utf=8=sig")


# ── Evaluate ─────────────────────────────────────────────────────
gemma4e4b_preds = preds

results = evaluate_model("gemma4-e4b-it_P2fewshot", true_labels, preds)
all_results.append(results)

Progress: 20/309
Progress: 40/309
Progress: 60/309
Progress: 80/309
Progress: 100/309
Progress: 120/309
Progress: 140/309
Progress: 160/309
Progress: 180/309
Progress: 200/309
Progress: 220/309
Progress: 240/309
Progress: 260/309
Progress: 280/309
Progress: 300/309

  gemma4-e4b-it_P2fewshot_think
                     precision    recall  f1-score   support

non-description (0)      0.725     0.955     0.825       155
    description (1)      0.933     0.636     0.757       154

           accuracy                          0.796       309
          macro avg      0.829     0.796     0.791       309
       weighted avg      0.829     0.796     0.791       309

Confusion matrix (rows=true, cols=pred):
  TN=148  FP=7
  FN=56  TP=98

95% Bootstrap Confidence Intervals (1000 resamples):
  f1_macro    : 0.790  [0.743 – 0.834]
  accuracy    : 0.796  [0.751 – 0.838]
  precision   : 0.830  [0.792 – 0.866]
  recall      : 0.796  [0.754 – 0.835]


# LLAMA via Ollama

In [7]:
# ============================================================
# LLAMA 3.1 8B Zeroshot
# ============================================================

import ollama
import pandas as pd

MODEL_NAME = "llama3.1:8b"

PROMPT_2 = """
I am attaching sentences from early modern British plays. You are a specialist in early modern British plays (c. 1500–1700). 
Your task is to Label each sentence according to whether it contains a description of an emotion according to the following 3 rules:
1. Label 1 for sentences explicitly describes an emotional state, e.g. "she felt grief", "he was seized with joy". 
2. Label 1 for sentences describing or metaphorically expressing an emotional state without naming an emotion, 
e.g. "piercing his heart", "passion overruling him".
3. Label 0 for sentences that do not describe an emotion, even if they depict an emotionally charged event or situation.
The examples in the rules are non-exhaustive. 
Return only the label 1 or 0. No explanation."""

# ── Prompt builder ───────────────────────────────────────────────
def build_messages(text, few_shot=False):
    messages = [{"role": "system", "content": PROMPT_2}]
    #if few_shot:
        #for example in FEW_SHOT_EXAMPLES:
            #messages.append({"role": "user",      "content": example["user"]})
            #messages.append({"role": "assistant", "content": example["assistant"]})
    messages.append({"role": "user", "content": text.strip()})

    return messages
    
# ── Predict ──────────────────────────────────────────────────────
df_results = df.copy()
preds, raws = [], []

def predict(texts, few_shot=False):
    for i, text in enumerate(texts): 
        messages = build_messages(text, few_shot=few_shot)
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
                #"num_predict": 3,       # equivalent to max_new_tokens=3
                "temperature": 0.0,     # deterministic output
            }
        )
        # full raw response content
        raws.append(response)
        # parse 0/1 from response
        answer = response["message"].content

        preds.append(
            1 if answer.startswith("1") else
            0 if answer.startswith("0") else
            (1 if "1" in answer[:10] else 0)
        )
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{len(texts)}")
    return preds, raws

# Main Loop
preds, raws = predict(texts, few_shot=False)

df_results.loc[df_results.index[:len(preds)], "preds"] = preds
df_results.loc[df_results.index[:len(raws)], "raws"] = raws

df_results.to_csv("llama3.1-8b_P2zeroshot.csv")


# ── Evaluate ─────────────────────────────────────────────────────
llama318B_preds = preds

results = evaluate_model("llama3.1-8b", true_labels, preds)
all_results.append(results)

Progress: 20/309
Progress: 40/309
Progress: 60/309
Progress: 80/309
Progress: 100/309
Progress: 120/309
Progress: 140/309
Progress: 160/309
Progress: 180/309
Progress: 200/309
Progress: 220/309
Progress: 240/309
Progress: 260/309
Progress: 280/309
Progress: 300/309

  llama3.1-8b
                     precision    recall  f1-score   support

non-description (0)      0.668     0.910     0.770       155
    description (1)      0.857     0.545     0.667       154

           accuracy                          0.728       309
          macro avg      0.763     0.728     0.719       309
       weighted avg      0.762     0.728     0.719       309

Confusion matrix (rows=true, cols=pred):
  TN=141  FP=14
  FN=70  TP=84

95% Bootstrap Confidence Intervals (1000 resamples):
  f1_macro    : 0.718  [0.668 – 0.767]
  accuracy    : 0.728  [0.680 – 0.777]
  precision   : 0.763  [0.716 – 0.808]
  recall      : 0.728  [0.682 – 0.773]


# Qwen, Deepseek via Ollama

In [10]:
# ============================================================
# QWEN using Ollama
# ============================================================

import ollama
import pandas as pd

MODEL_NAME = "qwen2.5:14b"

PROMPT_2 = """
I am attaching sentences from early modern British plays. You are a specialist in early modern British plays (c. 1500–1700). 
Your task is to Label each sentence according to whether it contains a description of an emotion according to the following 3 rules:
1. Label 1 for sentences explicitly describes an emotional state, e.g. "she felt grief", "he was seized with joy". 
2. Label 1 for sentences describing or metaphorically expressing an emotional state without naming an emotion, 
e.g. "piercing his heart", "passion overruling him".
3. Label 0 for sentences that do not describe an emotion, even if they depict an emotionally charged event or situation.
The examples in the rules are non-exhaustive. 
Return only the label 1 or 0. No explanation."""

# ── Prompt builder ───────────────────────────────────────────────
def build_messages(text, few_shot=False):
    messages = [{"role": "system", "content": PROMPT_2}]
    #if few_shot:
        #for example in FEW_SHOT_EXAMPLES:
            #messages.append({"role": "user",      "content": example["user"]})
            #messages.append({"role": "assistant", "content": example["assistant"]})
    messages.append({"role": "user", "content": text.strip()})

    return messages
    
# ── Predict ──────────────────────────────────────────────────────
df_results = df.copy()
preds, raws = [], []

def predict(texts, few_shot=False):
    for i, text in enumerate(texts): 
        messages = build_messages(text, few_shot=few_shot)
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
                #"num_predict": 3,       # equivalent to max_new_tokens=3
                "temperature": 0.0,     # deterministic output
            }
        )
        # full raw response content
        raws.append(response)
        # parse 0/1 from response
        answer = response["message"].content

        preds.append(
            1 if answer.startswith("1") else
            0 if answer.startswith("0") else
            (1 if "1" in answer[:10] else 0)
        )
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{len(texts)}")
    return preds, raws

# Main Loop
preds, raws = predict(texts, few_shot=False)

df_results.loc[df_results.index[:len(preds)], "preds"] = preds
df_results.loc[df_results.index[:len(raws)], "raws"] = raws

df_results.to_csv("qwen2.5-14b_P2zeroshot.csv")


# ── Evaluate ─────────────────────────────────────────────────────
qwen2514b_preds = preds

results = evaluate_model("qwen2.5-14b", true_labels, preds)
all_results.append(results)

Progress: 20/309
Progress: 40/309
Progress: 60/309
Progress: 80/309
Progress: 100/309
Progress: 120/309
Progress: 140/309
Progress: 160/309
Progress: 180/309
Progress: 200/309
Progress: 220/309
Progress: 240/309
Progress: 260/309
Progress: 280/309
Progress: 300/309

  qwen2.5-14b
                     precision    recall  f1-score   support

non-description (0)      0.683     0.961     0.799       155
    description (1)      0.934     0.552     0.694       154

           accuracy                          0.757       309
          macro avg      0.809     0.757     0.746       309
       weighted avg      0.808     0.757     0.747       309

Confusion matrix (rows=true, cols=pred):
  TN=149  FP=6
  FN=69  TP=85

95% Bootstrap Confidence Intervals (1000 resamples):
  f1_macro    : 0.745  [0.691 – 0.791]
  accuracy    : 0.756  [0.706 – 0.799]
  precision   : 0.808  [0.765 – 0.847]
  recall      : 0.756  [0.710 – 0.796]


In [11]:
# ============================================================
# DeepSeek R1 7B using Ollama
# ============================================================
import ollama
import pandas as pd

MODEL_NAME = "deepseek-r1:7b" 

PROMPT_2 = """
I am attaching sentences from early modern British plays. You are a specialist in early modern British plays (c. 1500–1700). 
Your task is to Label each sentence according to whether it contains a description of an emotion according to the following 3 rules:
1. Label 1 for sentences explicitly describes an emotional state, e.g. "she felt grief", "he was seized with joy". 
2. Label 1 for sentences describing or metaphorically expressing an emotional state without naming an emotion, 
e.g. "piercing his heart", "passion overruling him".
3. Label 0 for sentences that do not describe an emotion, even if they depict an emotionally charged event or situation.
The examples in the rules are non-exhaustive. 
Return only the label 1 or 0. No explanation."""

# ── Prompt builder ───────────────────────────────────────────────
def build_messages_DS(text, few_shot=False):
    system = "/no_think\n" + PROMPT_2 if "deepseek-r1" in MODEL_NAME else PROMPT_2
    messages = [{"role": "system", "content": system}]
    messages.append({"role": "user", "content": text.strip()})
    return messages
    
# ── Predict ──────────────────────────────────────────────────────
df_results = df.copy()

preds, raws = [], []

def predict(texts, few_shot=False):
    for i, text in enumerate(texts): 
        messages = build_messages_DS(text, few_shot=few_shot)
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
                #"num_predict": 3,       # equivalent to max_new_tokens=3
                "temperature": 0.0,     # deterministic output
            }
        )
        # full raw response content
        full_text = response # fix: extract string here
        raws.append(full_text)
        # parse 0/1 from response
        answer = response["message"].content

        preds.append(
            1 if answer.startswith("1") else
            0 if answer.startswith("0") else
            (1 if "1" in answer[:10] else 0)
        )
        if (i + 1) % 20 == 0:
            print(f"Progress: {i+1}/{len(texts)}")
    return preds, raws

# Main Loop
preds, raws = predict(texts, few_shot=False)

df_results.loc[df_results.index[:len(preds)], "preds"] = preds
df_results.loc[df_results.index[:len(raws)], "raws"] = raws

df_results.to_csv("deepseek7b_P2zeroshot.csv")


# ── Evaluate ─────────────────────────────────────────────────────
deepseek7b_preds = preds
results = evaluate_model("deepseek7b", true_labels, deepseek7b_preds)
all_results.append(results)

Progress: 20/309
Progress: 40/309
Progress: 60/309
Progress: 80/309
Progress: 100/309
Progress: 120/309
Progress: 140/309
Progress: 160/309
Progress: 180/309
Progress: 200/309
Progress: 220/309
Progress: 240/309
Progress: 260/309
Progress: 280/309
Progress: 300/309

  deepseek7b
                     precision    recall  f1-score   support

non-description (0)      0.603     0.890     0.719       155
    description (1)      0.787     0.409     0.538       154

           accuracy                          0.650       309
          macro avg      0.695     0.650     0.629       309
       weighted avg      0.695     0.650     0.629       309

Confusion matrix (rows=true, cols=pred):
  TN=138  FP=17
  FN=91  TP=63

95% Bootstrap Confidence Intervals (1000 resamples):
  f1_macro    : 0.628  [0.572 – 0.684]
  accuracy    : 0.651  [0.595 – 0.706]
  precision   : 0.695  [0.636 – 0.750]
  recall      : 0.649  [0.603 – 0.698]


# Model Comparison Summary

In [12]:
# ============================================================
# FINAL COMPARISON TABLE
# ============================================================
print("\n\n" + "="*70)
print("  FINAL COMPARISON TABLE")
print("="*70)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df.insert(0, "rank", results_df.index + 1)

# Console display
display_cols = ["rank", "model", "accuracy", "f1_macro", "f1_class0",
                "f1_class1", "precision", "recall"]
print(results_df[display_cols].to_string(index=False, float_format="{:.3f}".format))

print("\nF1-macro with 95% Bootstrap CI:")
for _, row in results_df.iterrows():
    bar = "█" * int(row["f1_macro"] * 20)
    print(f"  {row['rank']}. {row['model']:<40} "
          f"{row['f1_macro']:.3f}  [{row['ci_f1_lo']:.3f}–{row['ci_f1_hi']:.3f}]  {bar}")

# Save to Excel with formatting
with pd.ExcelWriter("eval.xlsx", engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Comparison", index=False)




  FINAL COMPARISON TABLE
 rank                          model  accuracy  f1_macro  f1_class0  f1_class1  precision  recall
    1  gemma4-e4b-it_P2fewshot_think     0.796     0.791      0.825      0.757      0.829   0.796
    2                    qwen2.5-14b     0.757     0.746      0.799      0.694      0.809   0.757
    3                    llama3.1-8b     0.728     0.719      0.770      0.667      0.763   0.728
    4 Hartmann DistilRoBERTa Emotion     0.722     0.718      0.686      0.750      0.734   0.722
    5     SamLowe RoBERTa GoEmotions     0.650     0.650      0.667      0.633      0.652   0.650
    6                     deepseek7b     0.650     0.629      0.719      0.538      0.695   0.650
    7          llama3.1-8b_P1fewshot     0.647     0.604      0.735      0.473      0.757   0.646
    8          llama3.1-8b_P1fewshot     0.647     0.604      0.735      0.473      0.757   0.646

F1-macro with 95% Bootstrap CI:
  1. gemma4-e4b-it_P2fewshot_think            0.791  [0.74